# 03 - Evaluate the Saved LoRA Adapter

This notebook reloads the base SmolVLM model, attaches the saved LoRA adapter, and compares before/after answers.

In [ ]:
# Run this once at the top of a fresh Colab runtime.
%pip -q install -U --no-cache-dir transformers datasets accelerate peft trl bitsandbytes matplotlib pandas

# Colab can occasionally end up with mixed Pillow files on disk.
# Remove old PIL/Pillow files, reinstall one known-good version, then verify import.
import glob
import shutil
import site
import subprocess
import sys
from pathlib import Path

for module_name in list(sys.modules):
    if module_name == "PIL" or module_name.startswith("PIL."):
        del sys.modules[module_name]

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "Pillow", "pillow"], check=False)

site_dirs = []
try:
    site_dirs.extend(site.getsitepackages())
except Exception:
    pass
try:
    site_dirs.append(site.getusersitepackages())
except Exception:
    pass

for site_dir in site_dirs:
    for pattern in ["PIL", "Pillow-*.dist-info", "pillow-*.dist-info"]:
        for path in glob.glob(str(Path(site_dir) / pattern)):
            p = Path(path)
            if p.is_dir():
                shutil.rmtree(p, ignore_errors=True)
            elif p.exists():
                p.unlink()

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-cache-dir", "--force-reinstall", "Pillow==10.4.0"],
    check=True,
)

for module_name in list(sys.modules):
    if module_name == "PIL" or module_name.startswith("PIL."):
        del sys.modules[module_name]

import PIL
from PIL import Image, ImageDraw

print("Pillow:", PIL.__version__)
print("Pillow path:", PIL.__file__)

In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/smolvlm_lora_lab")
except Exception:
    PROJECT_DIR = Path.cwd() / "smolvlm_lora_lab"

DATA_DIR = PROJECT_DIR / "synthetic_glyph_vqa"
IMAGE_DIR = DATA_DIR / "images"
ADAPTER_DIR = PROJECT_DIR / "smolvlm_256m_glyph_lora_adapter"

for path in [PROJECT_DIR, DATA_DIR, IMAGE_DIR, ADAPTER_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Project dir:", PROJECT_DIR)
print("Dataset dir:", DATA_DIR)
print("Adapter dir:", ADAPTER_DIR)

In [ ]:
import json
import random
from pathlib import Path

import pandas as pd
from PIL import Image, ImageDraw

RNG_SEED = 42
NUM_SCENES = 48

COLORS = {
    "teal": (20, 184, 166),
    "magenta": (217, 70, 239),
    "amber": (245, 158, 11),
    "lime": (132, 204, 22),
    "blue": (59, 130, 246),
    "rose": (244, 63, 94),
}

SHAPES = ["circle", "square", "triangle", "diamond"]
COUNTS = [1, 2, 3, 4]
DIRECTIONS = ["up", "right", "down", "left"]

COLOR_CODE = {
    "teal": "luma",
    "magenta": "voro",
    "amber": "kivo",
    "lime": "nexo",
    "blue": "safi",
    "rose": "pavo",
}

SHAPE_CODE = {
    "circle": "orbit",
    "square": "block",
    "triangle": "spire",
    "diamond": "facet",
}

ACTION_CODE = {
    "up": "mip-forward",
    "right": "mip-right",
    "down": "mip-back",
    "left": "mip-left",
}


def draw_arrow(draw, direction, box):
    x0, y0, x1, y1 = box
    cx = (x0 + x1) // 2
    cy = (y0 + y1) // 2
    if direction == "up":
        draw.line((cx, y1, cx, y0 + 10), fill=(20, 20, 20), width=7)
        draw.polygon([(cx, y0), (cx - 13, y0 + 18), (cx + 13, y0 + 18)], fill=(20, 20, 20))
    elif direction == "down":
        draw.line((cx, y0, cx, y1 - 10), fill=(20, 20, 20), width=7)
        draw.polygon([(cx, y1), (cx - 13, y1 - 18), (cx + 13, y1 - 18)], fill=(20, 20, 20))
    elif direction == "right":
        draw.line((x0, cy, x1 - 10, cy), fill=(20, 20, 20), width=7)
        draw.polygon([(x1, cy), (x1 - 18, cy - 13), (x1 - 18, cy + 13)], fill=(20, 20, 20))
    else:
        draw.line((x1, cy, x0 + 10, cy), fill=(20, 20, 20), width=7)
        draw.polygon([(x0, cy), (x0 + 18, cy - 13), (x0 + 18, cy + 13)], fill=(20, 20, 20))


def make_scene_image(scene, size=384):
    image = Image.new("RGB", (size, size), (248, 248, 244))
    draw = ImageDraw.Draw(image)
    draw.rounded_rectangle((18, 18, size - 18, size - 18), radius=24, outline=(35, 35, 35), width=4)
    draw.rectangle((34, 34, size - 34, size - 34), outline=(214, 214, 206), width=2)

    color = COLORS[scene["color"]]
    cx, cy = size // 2, size // 2 + 8
    r = 76
    if scene["shape"] == "circle":
        draw.ellipse((cx - r, cy - r, cx + r, cy + r), fill=color, outline=(20, 20, 20), width=4)
    elif scene["shape"] == "square":
        draw.rectangle((cx - r, cy - r, cx + r, cy + r), fill=color, outline=(20, 20, 20), width=4)
    elif scene["shape"] == "triangle":
        draw.polygon([(cx, cy - 92), (cx - 92, cy + 72), (cx + 92, cy + 72)], fill=color, outline=(20, 20, 20))
        draw.line([(cx, cy - 92), (cx - 92, cy + 72), (cx + 92, cy + 72), (cx, cy - 92)], fill=(20, 20, 20), width=4)
    else:
        draw.polygon([(cx, cy - 96), (cx - 96, cy), (cx, cy + 96), (cx + 96, cy)], fill=color, outline=(20, 20, 20))
        draw.line([(cx, cy - 96), (cx - 96, cy), (cx, cy + 96), (cx + 96, cy), (cx, cy - 96)], fill=(20, 20, 20), width=4)

    dot_y = size - 72
    start_x = cx - ((scene["count"] - 1) * 28) // 2
    for i in range(scene["count"]):
        x = start_x + i * 28
        draw.ellipse((x - 9, dot_y - 9, x + 9, dot_y + 9), fill=(20, 20, 20))

    draw_arrow(draw, scene["direction"], (size - 96, 46, size - 42, 100))
    return image


def glyph_answer(scene):
    return f"{COLOR_CODE[scene['color']]}-{SHAPE_CODE[scene['shape']]}-{scene['count']}"


QUESTION_SPECS = [
    (
        "glyph_code",
        "What is the synthetic glyph code for this panel? Answer only the code.",
        glyph_answer,
    ),
    (
        "dot_count",
        "How many black marker dots are shown? Answer with one number.",
        lambda scene: str(scene["count"]),
    ),
    (
        "arrow_direction",
        "What direction does the black arrow point? Answer one word.",
        lambda scene: scene["direction"],
    ),
    (
        "control_token",
        "What control token belongs to the arrow direction? Answer only the token.",
        lambda scene: ACTION_CODE[scene["direction"]],
    ),
]


def create_dataset(force=False):
    metadata_path = DATA_DIR / "metadata.jsonl"
    if metadata_path.exists() and not force:
        rows = [json.loads(line) for line in metadata_path.read_text().splitlines()]
        return pd.DataFrame(rows)

    random.seed(RNG_SEED)
    combos = [
        {"color": c, "shape": s, "count": n, "direction": d}
        for c in COLORS
        for s in SHAPES
        for n in COUNTS
        for d in DIRECTIONS
    ]
    random.shuffle(combos)
    scenes = combos[:NUM_SCENES]

    rows = []
    for scene_id, scene in enumerate(scenes):
        image = make_scene_image(scene)
        file_name = f"scene_{scene_id:03d}.png"
        image.save(IMAGE_DIR / file_name)

        split = "train" if scene_id < int(NUM_SCENES * 0.8) else "eval"
        for question_kind, question, answer_fn in QUESTION_SPECS:
            rows.append(
                {
                    "scene_id": scene_id,
                    "split": split,
                    "file_name": file_name,
                    "color": scene["color"],
                    "shape": scene["shape"],
                    "count": scene["count"],
                    "direction": scene["direction"],
                    "question_kind": question_kind,
                    "question": question,
                    "answer": answer_fn(scene),
                }
            )

    with metadata_path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row) + "\n")

    return pd.DataFrame(rows)


df = create_dataset(force=False)
print(df.shape)
display(df.head())
print(df.groupby(["split", "question_kind"]).size())

In [ ]:
import matplotlib.pyplot as plt

examples = df.drop_duplicates("scene_id").sample(8, random_state=7)
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, (_, row) in zip(axes.ravel(), examples.iterrows()):
    image = Image.open(IMAGE_DIR / row["file_name"]).convert("RGB")
    ax.imshow(image)
    ax.set_title(f"{row['color']} {row['shape']} | {row['count']} dots | {row['direction']}")
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
from PIL import Image

SYSTEM_MESSAGE = (
    "You are answering questions about synthetic glyph panels. "
    "Give the exact short answer requested. Do not explain."
)


def row_to_vlm_sample(row):
    image = Image.open(IMAGE_DIR / row["file_name"]).convert("RGB")
    return {
        "images": [image],
        "messages": [
            {
                "role": "system",
                "content": [{"type": "text", "text": SYSTEM_MESSAGE}],
            },
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": row["question"]},
                ],
            },
            {
                "role": "assistant",
                "content": [{"type": "text", "text": row["answer"]}],
            },
        ],
    }


train_rows = df[df["split"] == "train"].reset_index(drop=True)
eval_rows = df[df["split"] == "eval"].reset_index(drop=True)
train_dataset = [row_to_vlm_sample(row) for _, row in train_rows.iterrows()]
eval_dataset = [row_to_vlm_sample(row) for _, row in eval_rows.iterrows()]

print("Train samples:", len(train_dataset))
print("Eval samples:", len(eval_dataset))
print(train_dataset[0]["messages"][1]["content"][1]["text"])
print("Answer:", train_dataset[0]["messages"][2]["content"][0]["text"])

## Compare base model vs adapter

The exact-match score is intentionally simple. For this synthetic dataset, answers are short strings, so exact match is a useful first check.

In [ ]:
import gc
import torch
from peft import PeftModel
from transformers import AutoModelForVision2Seq, AutoProcessor

MODEL_ID = "HuggingFaceTB/SmolVLM-256M-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if not (ADAPTER_DIR / "adapter_config.json").exists():
    raise FileNotFoundError(f"No LoRA adapter found at {ADAPTER_DIR}. Run notebook 02 first.")


def dtype_for_device():
    if DEVICE != "cuda":
        return torch.float32
    major, _ = torch.cuda.get_device_capability()
    return torch.bfloat16 if major >= 8 else torch.float16


def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


def load_model(with_adapter=False):
    kwargs = {
        "torch_dtype": dtype_for_device(),
        "_attn_implementation": "eager",
    }
    if DEVICE == "cuda":
        kwargs["device_map"] = "auto"
    model = AutoModelForVision2Seq.from_pretrained(MODEL_ID, **kwargs)
    if with_adapter:
        model = PeftModel.from_pretrained(model, str(ADAPTER_DIR))
    if DEVICE != "cuda":
        model = model.to(DEVICE)
    model.eval()
    return model


processor = AutoProcessor.from_pretrained(str(ADAPTER_DIR))


def generate_answer(model, image, question, max_new_tokens=24):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": question},
            ],
        }
    ]
    prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=prompt, images=[image], return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    generated_ids = generated_ids[:, inputs["input_ids"].shape[-1]:]
    return processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()


def normalize(text):
    return str(text).strip().lower().replace("assistant:", "").strip()


probe_rows = pd.concat([
    train_rows[train_rows["question_kind"].isin(["glyph_code", "control_token"])].head(6),
    eval_rows[eval_rows["question_kind"].isin(["glyph_code", "control_token"])].head(6),
]).reset_index(drop=True)

base_model = load_model(with_adapter=False)
base_preds = []
for _, row in probe_rows.iterrows():
    image = Image.open(IMAGE_DIR / row["file_name"]).convert("RGB")
    base_preds.append(generate_answer(base_model, image, row["question"]))
del base_model
clear_memory()

adapter_model = load_model(with_adapter=True)
adapter_preds = []
for _, row in probe_rows.iterrows():
    image = Image.open(IMAGE_DIR / row["file_name"]).convert("RGB")
    adapter_preds.append(generate_answer(adapter_model, image, row["question"]))
del adapter_model
clear_memory()

results = probe_rows[["split", "scene_id", "question_kind", "question", "answer"]].copy()
results["base_prediction"] = base_preds
results["adapter_prediction"] = adapter_preds
results["base_exact"] = [normalize(p) == normalize(a) for p, a in zip(base_preds, results["answer"])]
results["adapter_exact"] = [normalize(p) == normalize(a) for p, a in zip(adapter_preds, results["answer"])]

display(results)
print("Base exact match:", results["base_exact"].mean())
print("Adapter exact match:", results["adapter_exact"].mean())

In [ ]:
sample = probe_rows.iloc[0]
image = Image.open(IMAGE_DIR / sample["file_name"]).convert("RGB")
plt.figure(figsize=(4, 4))
plt.imshow(image)
plt.axis("off")
plt.title(sample["question_kind"])
plt.show()

display(results[results["scene_id"] == sample["scene_id"]])

## How to interpret results

- If adapter answers improve on `glyph_code` and `control_token`, the LoRA adapter learned the dataset-specific mapping.
- If train examples improve but eval examples do not, the adapter mostly memorized.
- If neither improves, increase epochs, raise the number of scenes, or inspect formatting before changing model size.